# 01 — Data Cleaning

Cleans the NYC automated traffic-volume and motor-vehicle-collision datasets for the four-borough study area: Manhattan, Bronx, Brooklyn, and Queens.

**Analysis window:** 2022–2026, 07:00–23:59.

Traffic is retained as a supplementary exposure dataset because coverage is sparse across candidate bicycle segments. Crash events are retained for the core safety analysis.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Works when the notebook is run from either the repository root or /notebooks.
project = Path.cwd().resolve()
if project.name == "notebooks":
    project = project.parent

raw = project / "data" / "raw"
interim = project / "data" / "interim"
processed = project / "data" / "processed"
tables_dir = project / "outputs" / "tables"

interim.mkdir(parents=True, exist_ok=True)
processed.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project)

## Automated traffic-volume counts

In [ ]:
traffic_file = raw / "Automated_Traffic_Volume_Counts_20260905.csv"

usecols = [
    "RequestID", "Boro", "Yr", "M", "D",
    "HH", "MM", "Vol", "SegmentID", "WktGeom",
    "street", "fromSt", "toSt", "Direction"
]

traffic_chunks = []

for chunk in pd.read_csv(
    traffic_file,
    usecols=usecols,
    chunksize=200_000,
    low_memory=False
):
    chunk = chunk[
        chunk["Yr"].between(2022, 2026) &
        chunk["Boro"].isin(["Bronx", "Brooklyn", "Manhattan", "Queens"])
    ].copy()
    traffic_chunks.append(chunk)

traffic = pd.concat(traffic_chunks, ignore_index=True)

traffic["Vol"] = pd.to_numeric(traffic["Vol"], errors="coerce")

# Remove duplicate 15-minute observations.
traffic = traffic.drop_duplicates(
    subset=[
        "RequestID", "SegmentID", "Yr", "M", "D",
        "Direction", "HH", "MM"
    ]
)

print("Filtered traffic rows:", len(traffic))
print("Unique SegmentIDs:", traffic["SegmentID"].nunique())

In [ ]:
def assign_time_group(hour):
    if 7 <= hour <= 11:
        return "morning"
    if 12 <= hour <= 17:
        return "afternoon"
    if 18 <= hour <= 23:
        return "evening"
    return None

traffic["time_group"] = traffic["HH"].apply(assign_time_group)
traffic = traffic[traffic["time_group"].notna()].copy()

traffic_periods = (
    traffic
    .groupby(
        [
            "RequestID", "Boro", "Yr", "M", "D",
            "SegmentID", "WktGeom", "street", "fromSt", "toSt",
            "Direction", "time_group"
        ],
        dropna=False
    )
    .agg(
        period_volume=("Vol", "sum"),
        intervals_n=("Vol", "count")
    )
    .reset_index()
)

expected_intervals = {"morning": 20, "afternoon": 24, "evening": 24}
traffic_periods["expected_intervals"] = traffic_periods["time_group"].map(expected_intervals)
traffic_periods["complete_period"] = (
    traffic_periods["intervals_n"] == traffic_periods["expected_intervals"]
)

print(traffic_periods["complete_period"].value_counts())

`complete_period` is retained here as a QA diagnostic. The original analysis required observations in all three time groups but did not require every expected 15-minute interval to be present.

In [ ]:
traffic_period_totals = (
    traffic_periods
    .groupby(
        [
            "RequestID", "Boro", "Yr", "M", "D",
            "SegmentID", "WktGeom", "street", "fromSt", "toSt",
            "time_group"
        ],
        dropna=False
    )
    .agg(
        total_period_volume=("period_volume", "sum"),
        directions_n=("Direction", "nunique")
    )
    .reset_index()
)

index_cols = [
    "RequestID", "Boro", "Yr", "M", "D",
    "SegmentID", "WktGeom", "street", "fromSt", "toSt"
]

traffic_daily = (
    traffic_period_totals
    .pivot_table(
        index=index_cols,
        columns="time_group",
        values="total_period_volume",
        aggfunc="sum"
    )
    .reset_index()
)
traffic_daily.columns.name = None

direction_counts = (
    traffic_period_totals
    .groupby(index_cols, dropna=False)["directions_n"]
    .max()
    .reset_index()
)

traffic_daily = traffic_daily.merge(
    direction_counts,
    on=index_cols,
    how="left"
)

traffic_daily["direction_coverage"] = traffic_daily["directions_n"].map(
    {1: "one_direction", 2: "two_directions", 3: "three_directions"}
)

traffic_daily = traffic_daily[
    traffic_daily[["morning", "afternoon", "evening"]]
    .notna()
    .all(axis=1)
].copy()

traffic_daily["total_7_24"] = (
    traffic_daily["morning"] +
    traffic_daily["afternoon"] +
    traffic_daily["evening"]
)

# Zero-total records were treated as likely data artifacts.
traffic_daily_final = traffic_daily[
    traffic_daily["total_7_24"] > 0
].copy()

traffic_daily_final["Date"] = pd.to_datetime(
    dict(
        year=traffic_daily_final["Yr"],
        month=traffic_daily_final["M"],
        day=traffic_daily_final["D"]
    )
)

traffic_daily_final = traffic_daily_final[
    [
        "RequestID", "Boro", "Date", "Yr", "M", "D",
        "SegmentID", "WktGeom", "street", "fromSt", "toSt",
        "morning", "afternoon", "evening", "total_7_24",
        "directions_n", "direction_coverage"
    ]
]

print("Final traffic rows:", len(traffic_daily_final))
print(traffic_daily_final[["morning", "afternoon", "evening", "total_7_24"]].describe())

In [ ]:
traffic_out = processed / "traffic_daily_2022_2026.csv"
traffic_daily_final.to_csv(traffic_out, index=False)
print("Saved:", traffic_out)

## Motor-vehicle collisions

In [ ]:
crash_file = raw / "Motor_Vehicle_Collisions_-_Crashes_20260902.csv"
crashes = pd.read_csv(crash_file, low_memory=False)

crashes["CRASH DATE"] = pd.to_datetime(
    crashes["CRASH DATE"],
    errors="coerce"
)

crashes = crashes[
    crashes["CRASH DATE"].dt.year.between(2022, 2026)
].copy()

study_boros = ["MANHATTAN", "BRONX", "BROOKLYN", "QUEENS"]
crashes = crashes[crashes["BOROUGH"].isin(study_boros)].copy()

crashes["crash_hour"] = pd.to_numeric(
    crashes["CRASH TIME"].str.split(":").str[0],
    errors="coerce"
)
crashes["time_group"] = crashes["crash_hour"].apply(assign_time_group)
crashes = crashes[crashes["time_group"].notna()].copy()

crashes["valid_coords"] = (
    crashes["LATITUDE"].notna() &
    crashes["LONGITUDE"].notna() &
    (crashes["LATITUDE"] != 0) &
    (crashes["LONGITUDE"] != 0)
)

print("Filtered crash events:", len(crashes))
print(crashes["time_group"].value_counts())
print(crashes["valid_coords"].value_counts())

In [ ]:
crash_fields = [
    "COLLISION_ID", "CRASH DATE", "CRASH TIME", "time_group",
    "BOROUGH", "ZIP CODE", "LATITUDE", "LONGITUDE", "LOCATION",
    "valid_coords", "ON STREET NAME", "CROSS STREET NAME",
    "OFF STREET NAME", "NUMBER OF PERSONS INJURED",
    "NUMBER OF PERSONS KILLED", "NUMBER OF PEDESTRIANS INJURED",
    "NUMBER OF PEDESTRIANS KILLED", "NUMBER OF CYCLIST INJURED",
    "NUMBER OF CYCLIST KILLED", "NUMBER OF MOTORIST INJURED",
    "NUMBER OF MOTORIST KILLED"
]

crashes_clean = crashes[crash_fields].copy()

crash_out = processed / "crashes_clean_2022_2026.csv"
crashes_clean.to_csv(crash_out, index=False)

print("Saved:", crash_out)
print("Rows:", len(crashes_clean))